In [7]:
import os
import pandas as pd

from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# configuring conection
DB_USER = os.getenv("POSTGRES_USER", "admin")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")

if not DB_PASSWORD:
    raise ValueError(
        "POSTGRESS_PASSWPRD not found :(. "
        "Check .env you dump."
    )

url = URL.create(
    drivername="postgresql+psycopg",
    username=DB_USER,
    password=DB_PASSWORD,
    host="city_move",
    port=5432,
    database=os.getenv("CITYMOVE_DB", "city_move"),
)

engine = create_engine(url)

print("conection sucess! :).")

conection sucess! :).


In [8]:

# Query for SQL
query = """
SELECT
    f.id AS id_fahrt,
    f.id_linie,
    f.id_fahrzeug,
    f.datum,
    f.fahrtrichtung,
    fp.id_haltestelle,
    fp.ankunft_geplant,
    fh.ankunft_tatsaechlich,
    EXTRACT(
        EPOCH FROM (
            fh.ankunft_tatsaechlich - fp.ankunft_geplant
        )
    ) / 60.0 AS verspaetung_minuten
FROM citymove.fahrplan AS fp
INNER JOIN citymove.fahrt_haltestelle AS fh
    ON fp.id_fahrt = fh.id_fahrt
    AND fp.id_haltestelle = fh.id_haltestelle
INNER JOIN citymove.fahrt AS f
    ON f.id = fp.id_fahrt
WHERE
    fp.ankunft_geplant IS NOT NULL
    AND fh.ankunft_tatsaechlich IS NOT NULL
"""

# Save results in the DataFrame
df = pd.read_sql_query(query, engine)

# Show results
print("Registries:", len(df))
display(df.head())

Registries: 2065


,id_fahrt,id_linie,id_fahrzeug,datum,fahrtrichtung,id_haltestelle,ankunft_geplant,ankunft_tatsaechlich,verspaetung_minuten
0,1,10,14,2026-06-29,Rückfahrt,20,2026-06-29 14:15:00,2026-06-29 14:18:00,3.0
1,1,10,14,2026-06-29,Rückfahrt,14,2026-06-29 14:21:00,2026-06-29 14:21:00,0.0
2,1,10,14,2026-06-29,Rückfahrt,1,2026-06-29 14:24:00,2026-06-29 14:24:00,0.0
3,1,10,14,2026-06-29,Rückfahrt,9,2026-06-29 14:30:00,2026-06-29 14:38:00,8.0
4,1,10,14,2026-06-29,Rückfahrt,2,2026-06-29 14:35:00,2026-06-29 14:37:00,2.0


In [9]:


# Create a copy to preserve the original data
df_ml = df.copy()

# Convert date columns to datetime format
df_ml["ankunft_geplant"] = pd.to_datetime(
    df_ml["ankunft_geplant"]
)

df_ml["datum"] = pd.to_datetime(
    df_ml["datum"]
)

# Create time-based features
df_ml["programmed_hour"] = (
    df_ml["ankunft_geplant"].dt.hour
    + df_ml["ankunft_geplant"].dt.minute / 60
)

df_ml["day_of_week"] = (
    df_ml["ankunft_geplant"].dt.dayofweek
)

df_ml["month"] = (
    df_ml["ankunft_geplant"].dt.month
)

# Remove records without a target value or scheduled arrival time
df_ml = df_ml.dropna(
    subset=[
        "verspaetung_minuten",
        "ankunft_geplant"
    ]
)

# Display the results
print("Available records:", len(df_ml))

display(df_ml.head())



Available records: 2065


,id_fahrt,id_linie,id_fahrzeug,datum,fahrtrichtung,id_haltestelle,ankunft_geplant,ankunft_tatsaechlich,verspaetung_minuten,programmed_hour,day_of_week,month
0,1,10,14,2026-06-29,Rückfahrt,20,2026-06-29 14:15:00,2026-06-29 14:18:00,3.0,14.250000,0,6
1,1,10,14,2026-06-29,Rückfahrt,14,2026-06-29 14:21:00,2026-06-29 14:21:00,0.0,14.350000,0,6
2,1,10,14,2026-06-29,Rückfahrt,1,2026-06-29 14:24:00,2026-06-29 14:24:00,0.0,14.400000,0,6
3,1,10,14,2026-06-29,Rückfahrt,9,2026-06-29 14:30:00,2026-06-29 14:38:00,8.0,14.500000,0,6
4,1,10,14,2026-06-29,Rückfahrt,2,2026-06-29 14:35:00,2026-06-29 14:37:00,2.0,14.583333,0,6


In [10]:
print("Número de registros:", len(df_ml))
print("Número de viajes diferentes:", df_ml["id_fahrt"].nunique())
print("Número de líneas:", df_ml["id_linie"].nunique())
print("Número de vehículos:", df_ml["id_fahrzeug"].nunique())
print("Número de paradas:", df_ml["id_haltestelle"].nunique())

print("\nFechas disponibles:")
print("Desde:", df_ml["ankunft_geplant"].min())
print("Hasta:", df_ml["ankunft_geplant"].max())

Número de registros: 2065
Número de viajes diferentes: 213
Número de líneas: 10
Número de vehículos: 49
Número de paradas: 36

Fechas disponibles:
Desde: 2026-03-31 08:45:00
Hasta: 2026-09-24 08:23:00


In [11]:
import numpy as np
import pandas as pd

# Copy dataframe
data = df_ml.copy()

# erase data, that are not important
required_columns = [
    "id_linie",
    "id_fahrzeug",
    "id_haltestelle",
    "fahrtrichtung",
    "ankunft_geplant",
    "verspaetung_minuten",
]

data = data.dropna(subset=required_columns)

# temp vars
data["ankunft_geplant"] = pd.to_datetime(
    data["ankunft_geplant"]
)

data["programmed_hour"] = (
    data["ankunft_geplant"].dt.hour
    + data["ankunft_geplant"].dt.minute / 60
)

data["week_day"] = data["ankunft_geplant"].dt.dayofweek
data["mes"] = data["ankunft_geplant"].dt.month

# variablen in a pattern 
data = data.sort_values("ankunft_geplant").reset_index(drop=True)

print("Registry for the model:", len(data))
print(
    "First date:",
    data["ankunft_geplant"].min()
)
print(
    "Last date:",
    data["ankunft_geplant"].max()
)

display(data.head())

Registry for the model: 2065
First date: 2026-03-31 08:45:00
Last date: 2026-09-24 08:23:00


,id_fahrt,id_linie,id_fahrzeug,datum,fahrtrichtung,id_haltestelle,ankunft_geplant,ankunft_tatsaechlich,verspaetung_minuten,programmed_hour,day_of_week,month,week_day,mes
0,168,8,49,2026-03-31,Hinfahrt,31,2026-03-31 08:45:00,2026-03-31 08:45:00,0.0,8.750000,1,3,1,3
1,168,8,49,2026-03-31,Hinfahrt,29,2026-03-31 08:47:00,2026-03-31 08:48:00,1.0,8.783333,1,3,1,3
2,168,8,49,2026-03-31,Hinfahrt,6,2026-03-31 08:52:00,2026-03-31 09:00:00,8.0,8.866667,1,3,1,3
3,168,8,49,2026-03-31,Hinfahrt,11,2026-03-31 08:55:00,2026-03-31 08:55:00,0.0,8.916667,1,3,1,3
4,168,8,49,2026-03-31,Hinfahrt,5,2026-03-31 08:59:00,2026-03-31 09:00:00,1.0,8.983333,1,3,1,3


In [14]:
from sklearn.model_selection import train_test_split

# Variables for start
feature_columns = [
    "id_linie",
    "id_fahrzeug",
    "id_haltestelle",
    "fahrtrichtung",
    "programmed_hour",
    "week_day",
    "month",
]

X = data[feature_columns].copy()
y = data["verspaetung_minuten"].copy()

# separate the most new values with de datetime
split_index = int(len(data) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("training registries:", len(X_train))
print("testing registries:", len(X_test))

print(
    "last date for training:",
    data.iloc[split_index - 1]["ankunft_geplant"]
)

print(
    "first date to testing:",
    data.iloc[split_index]["ankunft_geplant"]
)

training registries: 1652
testing registries: 413
last date for training: 2026-08-11 11:30:00
first date to testing: 2026-08-11 11:34:00


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Variables categóricas
categorical_features = [
    "id_linie",
    "id_fahrzeug",
    "id_haltestelle",
    "fahrtrichtung",
]

# Variables numéricas
numeric_features = [
    "hora_programada",
    "dia_semana",
    "mes",
]

# Preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features,
        ),
    ]
)

# Modelo 1: referencia basada en la mediana
baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)

# Modelo 2: Random Forest
random_forest = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

random_forest.fit(X_train, y_train)

print("Entrenamiento terminado.")

ValueError: Some column names are not columns of the dataframe: {'mes', 'hora_programada', 'dia_semana'}